### Widok (logika)

Następnym krokiem jest zaimplementowanie funkcji, która obsłuży dane rejestrowanego użytkownika. Możemy w tym przypadku skorzystać z `UserCreationForm` czyli przygotowanego przez autorów Django formularza rejestracji.

Zaimportujmy go na początku pliku [`movies/views.py`](http://localhost:8888/edit/movies/views.py) dodając:

```python
from django.contrib.auth.forms import UserCreationForm
```

A następnie przejdźmy do tworzenia widoku:

```python
def user_signup(request):
    if request.method == 'POST':
        # tu trzeba przetworzyć dane z formularza
        pass
    else:
        # tutaj obsługujemy przypadek kiedy użytkownik pierwszy 
        # raz wyświetlił stronę
        form = UserCreationForm()
    
    # na końcu zwracamy wyrenderowanego HTMLa
    return render(
        request,
        template_name="registration/signup_form.html",
        context={'form': form}
    )
```

Dzieje się tutaj kilka rzeczy.

Jeśli zapytanie ma metodę "POST" oznacza to zwykle, że dostaliśmy jakieś dane zakodowane w zapytanie HTTP (ogromne
uproszczenie). Zwykle zapytań HTTP-POST używa się do przesłania formularzy, które to mają w efekcie coś nowego
stworzyć w bazie danych.

W tym przypadku tworzony będzie nowy użytkownik z danych formularza (`UserCreationForm`) przesłanych w zapytanie
HTTP z przeglądarki. `UserCreationForm` to coś co pozwala na wyświetlenie formularza w HTML oraz odwzorowanie danych
od użytkownika na login i hasło przyszłego użytkownika.

#### Dlaczego sprawdzamy `request.method == "POST"`?

Użytkownikowi, który pierwszy raz wszedł na stronę rejestracji chcemy wyświetlić pusty, "świeży" formularz rejestracji.
Przeglądarki domyślnie wysyłają zapytanie metodą [GET](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Methods/GET). Jest to typowe zapytanie przeglądarki "podaj mi zawartość tej
strony, chcę ją wyświetlić mojemu użytkownikowi".

Jeżeli użytkownik kliknął "rejestruj się" po wypełnieniu formularza - przeglądarka wyśle zapytanie metodą POST - Django odbierze dane formularza zakodowane w zapytaniu POST a następnie odpowiednio je przetworzy. Z tych przetworzonych danych będziemy mogli "stworzyć" nowego użytkownika w naszej bazie danych.

#### Szablony HTML

Dla porządku stwórzmy sobie dwa szablony HTML, jeden na formularz rejestracji i drugi na wyświetlenie podziękowania za rejestracje na naszej stronie.

Potrzebujemy więc:

In [13]:
!touch movies/templates/registration/signup_form.html

[`movies/templates/registration/signup_form.html`](http://localhost:8888/edit/movies/templates/registration/signup_form.html)

```django
{% extends "base.html" %}

{% block content %}
    <form method="post">
        {% csrf_token %}

        {{ form }}

        <hr>
        
        <button class="btn btn-primary">Zarejestruj się</button>
    </form>
{% endblock %}
```

Ten szablon HTML wyświetli nam formularz rejestracji (`{{ form }}`), który przekażemy w kontekście z widoku, który
właśnie piszemy. Formularz potrzebuje być otoczony specyficznym tagiem HTML `<form></form>`.

Ważne jest też dodanie metody wysyłki danych z formularza `<form method="post">` przez atrybut `method` oraz
oczywiście zamieszczenie przycisku, żeby można było zatwierdzić wpisane dane :)

`{% csrf_token %}` wygeneruje nam token CSRF. Jest to popularne zabezpieczenie przed atakami *Cross-site request forgery*.
Dla zainteresowanych polecam np. https://sekurak.pl/czym-jest-podatnosc-csrf-cross-site-request-forgery/

```zsh
python3 manage.py runserver
```

http://127.0.0.1:8000/accounts/signup/

Drugi szablon będzie znacznie prostszy:

In [14]:
!touch movies/templates/registration/signup_complete.html

[`movies/templates/registration/signup_complete.html`](http://localhost:8888/edit/movies/templates/registration/signup_complete.html)

```django
{% extends "base.html" %}

{% block content %}
    <h1>Dziękujemy za rejestrację na naszej stronie!</h1>

    <a href="{% url 'login' %}" class="btn btn-success">
        Zaloguj się
    </a>
{% endblock %}
```

#### Zapisywanie danych z formularza

Mamy dwa szablony oraz szkielet widoku. Jednak jeśli wypełnimy formularz i zatwierdzimy to użytkownik się nie tworzy.

Dla kodu `def user_signup(request):` z powyżej nawet dostajemy błąd. Dopiszmy więc brakujące przetwarzanie
formularza.

```python
    if request.method == 'POST':
        # tu trzeba przetworzyć dane z formularza
        form = UserCreationForm(request.POST)
        if form.is_valid():
            form.save()
            return render(
                request,
                template_name="registration/signup_complete.html"
            )
```

Uwaga na wcięcia! To wszystko ma się wykonać dla metody POST.

Przeanalizujmy. Wczytajmy do naszego formularza (`UserCreationForm`) dane z zapytania HTTP (`request.POST`) od
użytkownika:

```python
form = UserCreationForm(request.POST)
```

Następnie sprawdźmy, czy dane w formularzu są poprawne. W tym przypadku jest to porównanie hasła 1 i hasła 2,
sprawdzenie czy są wypełnione wszystkie pola itp. Następnie zapiszmy formularz tym samym tworząc nowego użytkownika
w bazie danych.

Tak działają formularze w Django - pozwalają nam wczytać dane od użytkownika i na ich bazie stworzyć nowy obiekt w
bazie danych. Wszystkim zajmuje się za kulisami Django. My tylko deklarujemy co i z czego.

#### Finalny widok rejestracji

Po wszystkich modyfikacjach widok rejestracji powinien wyglądać następująco:

```python
def user_signup(request):
    if request.method == 'POST':
        # tu trzeba przetworzyć dane z formularza
        form = UserCreationForm(request.POST)
        if form.is_valid():
            form.save()
            return render(
                request,
                template_name="registration/signup_complete.html"
            )
    else:
        # tutaj obsługujemy przypadek kiedy użytkownik pierwszy
        # raz wyświetlił stronę
        form = UserCreationForm()

    # na końcu zwracamy wyrenderowanego HTMLa
    return render(
        request,
        template_name="registration/signup_form.html",
        context={'form': form}
    )
```

Spróbujmy się teraz zarejestrować, a następnie zalogować na nowe konto :)

http://127.0.0.1:8000/accounts/signup/

## Dalsze kroki

### Ćwiczenie

Warto również dodać swój szablon dla strony wylogowania tworząc szablon:

In [15]:
!touch movies/templates/registration/logged_out.html

[`movies/templates/registration/logged_out.html`](http://localhost:8888/edit/movies/templates/registration/logged_out.html)

Uwaga: żeby pomyślnie nadpisać szablon dostarczany przez bibliotekę, taki jak widok wylogowania się, trzeba upewnić się, że nasza aplikacja jest przez panelem administracyjnym w `INSTALLED_APPS` ([`goodmovies/settings.py`](http://localhost:8888/edit/goodmovies/settings.py)).

```python
INSTALLED_APPS = [
    'movies',  # <-- PRZED django.contrib.admin
    'django.contrib.admin',  # django.contrib.admin
    'django.contrib.auth',
    'django.contrib.contenttypes',
    'django.contrib.sessions',
    'django.contrib.messages',
    'django.contrib.staticfiles',
]
```

Dzięki temu szablony z naszej aplikacji `movies` będą miały priorytet nad wszystkimi dostarczonymi przez pozostałe
aplikacje :) W szczególności pozwoli nam to na dostosowanie wyglądu panelu administracyjnego i inne.

Tutaj przyda nam się tag `{% url 'nazwa_widoku' %}` oraz lista dostępnych: https://docs.djangoproject.com/en/5.2/topics/auth/default/#module-django.contrib.auth.views

### Rozwiązanie

#### Krok 1 – Upewnij się, że `movies` jest na początku `INSTALLED_APPS`

W pliku [`goodmovies/settings.py`](http://localhost:8888/edit/goodmovies/settings.py):

```python
INSTALLED_APPS = [
    'movies',  # <-- musi być PRZED django.contrib.admin
    'django.contrib.admin',
    'django.contrib.auth',
    'django.contrib.contenttypes',
    'django.contrib.sessions',
    'django.contrib.messages',
    'django.contrib.staticfiles',
]
```

#### Krok 2 - Utwórz szablon [`movies/templates/registration/logged_out.html`](http://localhost:8888/edit/movies/templates/registration/logged_out.html)

W terminalu lub ręcznie:

In [16]:
!touch movies/templates/registration/logged_out.html

#### Krok 3 - Zawartość [`movies/templates/registration/logged_out.html`](http://localhost:8888/edit/movies/templates/registration/logged_out.html)

Plik: [`movies/templates/registration/logged_out.html`](http://localhost:8888/edit/movies/templates/registration/logged_out.html)

Do utworzonego pliku dodaj własny szablon:

```django
{% extends "base.html" %}

{% block content %}

    <h1>Do zobaczenia!</h1>
    <p>Wróć do nas wkrótce.</p>
    <p><a href="{% url 'login' %}">Zaloguj się ponownie</a></p>

{% endblock %}
```

#### Krok 4 - Popraw szablon [`movies/templates/profile.html`](http://localhost:8888/edit/movies/templates/profile.html)

Umożliwi to wylogowanie się z poziomu profilu użytkownika:

```django
{% extends "base.html" %}

{% block content %}

    <h1>Witaj {{ request.user.username }}!</h1>
    <p>Ostatnie logowanie {{ request.user.last_login }}</p>

    <form method="post" action="{% url 'logout' %}">
        {% csrf_token %}
        <button type="submit">Wyloguj się</button>
    </form>

{% endblock %}
```

#### Krok 5 - Przetestuj

1. Zaloguj się (np. przez http://127.0.0.1:8000/accounts/login/).

2. Wejdź na http://127.0.0.1:8000/accounts/profile/ - powinien wyświetlić się Twój profil z przyciskiem **Wyloguj się**.

3. Kliknij przycisk **Wyloguj się**.

4. Powinien pojawić się Twój własny szablon [`movies/templates/registration/logged_out.html`](http://localhost:8888/edit/movies/templates/registration/logged_out.html) z linkiem do ponownego logowania i strony głównej.

#### Gotowe!

Masz teraz pełne wsparcie dla logowania i wylogowania z własnymi szablonami, zgodnie z zasadami Django.

## Rozszerzamy interfejs użytkownika

### Ćwiczenie

W tej chwili linki do logowania i wylogowania użytkownika nie są umieszczone nigdzie w interfejsie. Poprawmy więc UX (user-experience) naszej strony dodając odpowiednie linki w pasku nawigacji.

### Rozwiązanie

#### Problem

Strona nie zawiera nigdzie widocznych **linków do logowania i wylogowania**, co utrudnia korzystanie z aplikacji użytkownikowi końcowemu. Należy dodać te odnośniki w widocznym miejscu (np. nagłówku strony), dostosowując ich widoczność w zależności od stanu zalogowania.

#### Krok 1 - Edytuj plik bazowy szablonu (np. [`movies/templates/base.html`](http://localhost:8888/edit/movies/templates/base.html))

```django
<!DOCTYPE html>
<html lang="pl">
<head>
    <meta charset="UTF-8">
    <title>Moja biblioteka filmów</title>
</head>
<body>
    
    {% if request.user.is_authenticated %}
        <p>Zalogowany jako {{ request.user.username }}</p>
        <form method="post" action="{% url 'logout' %}">
            {% csrf_token %}
            <button type="submit">Wyloguj się</button>
        </form>
    {% else %}
        <p><a href="{% url 'login' %}">Zaloguj się</a></p>
    {% endif %}
    
    <h1>
        Witamy w filmotece!
    </h1>
    {% block content %}
        Tu będzie treść...
    {% endblock %}
</body>
</html>

```

#### Wyjaśnienie

* `request.user.is_authenticated` - sprawdza, czy użytkownik jest zalogowany.

* `{% url 'login' %}` - kieruje do domyślnego widoku logowania Django.

* `{% url 'logout' %}` - kieruje do widoku wylogowania (musimy użyć POST!).

* `csrf_token` - wymagany w formularzu POST.

#### Krok 2 - Przetestuj

1. Odwiedź stronę z nagłówkiem (np. http://127.0.0.1:8000/hello/)

2. Zobaczysz link **Zaloguj się**, gdy nie jesteś zalogowany.

3. Po zalogowaniu pojawi się komunikat z nazwą użytkownika i przyciskiem **Wyloguj**.

4. Wylogowanie przeniesie Cię do wcześniej utworzonego szablonu [`movies/templates/registration/logged_out.html`](http://localhost:8888/edit/movies/templates/registration/logged_out.html).

## Źródła i materiały pomocnicze

* https://docs.djangoproject.com/en/5.2/topics/auth/

* https://docs.djangoproject.com/en/5.2/ref/contrib/auth/#django.contrib.auth.models.User

* https://docs.djangoproject.com/en/5.2/ref/templates/builtins/

* https://learndjango.com/tutorials/django-signup-tutorial